# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mohidraheel/Machine-Learning-Practice/blob/main/work/notebooks/w03_feature_leakage_check.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
!pip -q install duckdb huggingface_hub pandas numpy

In [2]:
import os
import json
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd

from google.colab import userdata
from huggingface_hub import login
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 180)

print("Libraries loaded.")

Libraries loaded.


In [3]:
hf_token = userdata.get("HF_TOKEN")

if not hf_token:
    raise ValueError(
        "HF_TOKEN was not found. Add it in Colab Secrets and enable notebook access."
    )

os.environ["HF_TOKEN"] = hf_token
login(token=hf_token, add_to_git_credential=False)

print("Hugging Face authentication completed.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face authentication completed.


In [4]:
con = duckdb.connect()

con.execute("INSTALL httpfs")
con.execute("LOAD httpfs")
con.execute("SET secret_directory='/tmp'")

con.execute(f'''
    CREATE OR REPLACE SECRET hf_secret (
        TYPE HUGGINGFACE,
        TOKEN '{hf_token}'
    )
''')

WAREHOUSE_PATH = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/**/*.parquet"
)

con.execute(f'''
    CREATE OR REPLACE VIEW warehouse AS
    SELECT *
    FROM read_parquet(
        '{WAREHOUSE_PATH}',
        hive_partitioning=true
    )
''')

print("Warehouse view created.")

Warehouse view created.


In [5]:
schema_df = con.execute("DESCRIBE warehouse").df()
display(schema_df)

required_columns = [
    "report_date",
    "client_hash_id",
    "content_hash_id",
    "gsc_data_available",
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_data_available",
    "ga4_sessions",
]

available_columns = schema_df["column_name"].tolist()

missing_columns = [
    column for column in required_columns
    if column not in available_columns
]

if missing_columns:
    raise KeyError(f"Missing required warehouse columns: {missing_columns}")

print("Required warehouse columns are available.")

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


Required warehouse columns are available.


## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

The source is aggregated to one row per anonymized client-content pair for March 2026.

Engineered features:

- Log impressions
- Log clicks
- CTR
- Impression-weighted position
- Log sessions
- GA4 availability indicator
- One-hot encoded position bucket

Pages with fewer than 100 impressions are excluded because CTR can be unstable at very low volume.

In [6]:
feature_source_sql = '''
SELECT
    client_hash_id AS client_id,
    content_hash_id AS content_id,

    SUM(COALESCE(gsc_impressions, 0)) AS impressions_31d,
    SUM(COALESCE(gsc_clicks, 0)) AS clicks_31d,

    CASE
        WHEN SUM(COALESCE(gsc_impressions, 0)) > 0
        THEN SUM(COALESCE(gsc_clicks, 0)) * 1.0
             / SUM(COALESCE(gsc_impressions, 0))
        ELSE NULL
    END AS ctr_31d,

    CASE
        WHEN SUM(
            CASE
                WHEN gsc_avg_position IS NOT NULL
                THEN COALESCE(gsc_impressions, 0)
                ELSE 0
            END
        ) > 0
        THEN SUM(
            CASE
                WHEN gsc_avg_position IS NOT NULL
                THEN gsc_avg_position * COALESCE(gsc_impressions, 0)
                ELSE 0
            END
        ) * 1.0
        / SUM(
            CASE
                WHEN gsc_avg_position IS NOT NULL
                THEN COALESCE(gsc_impressions, 0)
                ELSE 0
            END
        )
        ELSE NULL
    END AS weighted_position_31d,

    SUM(COALESCE(ga4_sessions, 0)) AS sessions_31d,
    MAX(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS has_ga4_data,
    COUNT(*) AS source_rows

FROM warehouse

WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
  AND gsc_data_available IS TRUE

GROUP BY
    client_hash_id,
    content_hash_id

HAVING SUM(COALESCE(gsc_impressions, 0)) >= 100
'''

raw_features = con.execute(feature_source_sql).df()

print("Eligible rows:", len(raw_features))
display(raw_features.head(10))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Eligible rows: 101441


,client_id,content_id,impressions_31d,clicks_31d,ctr_31d,weighted_position_31d,sessions_31d,has_ga4_data,source_rows
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,6523.0,7.0,0.001073,6.893301,1.0,1,31
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,453.0,0.0,0.000000,3.214128,0.0,0,31
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,5630.0,6.0,0.001066,6.535346,3.0,1,31
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,4944.0,13.0,0.002629,7.435680,2.0,1,31
4,client_73cda7b4e4f265ea,content_1855a661b4d36130,429.0,1.0,0.002331,3.871795,2.0,1,31
5,client_73cda7b4e4f265ea,content_5d412fba6e1a2582,223.0,1.0,0.004484,10.538117,2.0,1,31
6,client_73cda7b4e4f265ea,content_22c063002b7c1caf,314.0,1.0,0.003185,10.057325,0.0,0,31
7,client_73cda7b4e4f265ea,content_aafb2ab7e5fc80d0,7709.0,20.0,0.002594,5.127643,12.0,1,31
8,client_73cda7b4e4f265ea,content_20403327d8d9374c,3561.0,10.0,0.002808,9.273799,22.0,1,31
9,client_73cda7b4e4f265ea,content_f71459b346aba398,175.0,1.0,0.005714,38.560000,0.0,0,29


In [7]:
assert len(raw_features) > 0, "No eligible rows were returned."
assert raw_features[["client_id", "content_id"]].notna().all().all()
assert raw_features["impressions_31d"].ge(100).all()
assert raw_features["ctr_31d"].dropna().between(0, 1).all()

feature_work = raw_features.copy()

feature_work["log_impressions_31d"] = np.log1p(
    feature_work["impressions_31d"]
)
feature_work["log_clicks_31d"] = np.log1p(
    feature_work["clicks_31d"]
)
feature_work["log_sessions_31d"] = np.log1p(
    feature_work["sessions_31d"]
)

position_bins = [0, 3, 5, 10, 20, 50, np.inf]
position_labels = ["1-3", "4-5", "6-10", "11-20", "21-50", "51+"]

feature_work["position_bucket"] = pd.cut(
    feature_work["weighted_position_31d"],
    bins=position_bins,
    labels=position_labels,
    include_lowest=True,
).astype("object")

feature_work["position_bucket"] = (
    feature_work["position_bucket"].fillna("Unknown")
)

fill_values = {
    "log_impressions_31d": 0.0,
    "log_clicks_31d": 0.0,
    "ctr_31d": 0.0,
    "weighted_position_31d": float(
        feature_work["weighted_position_31d"].median()
    ),
    "log_sessions_31d": 0.0,
    "has_ga4_data": 0,
}

for column, value in fill_values.items():
    feature_work[column] = feature_work[column].fillna(value)

numeric_features = [
    "log_impressions_31d",
    "log_clicks_31d",
    "ctr_31d",
    "weighted_position_31d",
    "log_sessions_31d",
    "has_ga4_data",
]

categorical_features = pd.get_dummies(
    feature_work[["position_bucket"]],
    prefix="position",
    dtype=int,
)

X = pd.concat(
    [
        feature_work[numeric_features].reset_index(drop=True),
        categorical_features.reset_index(drop=True),
    ],
    axis=1,
)

identifiers = feature_work[
    ["client_id", "content_id"]
].reset_index(drop=True)

feature_vector = pd.concat(
    [identifiers, X],
    axis=1,
)

print("Feature-vector shape:", feature_vector.shape)
display(feature_vector.head(10))

Feature-vector shape: (101441, 14)


,client_id,content_id,log_impressions_31d,log_clicks_31d,ctr_31d,weighted_position_31d,log_sessions_31d,has_ga4_data,position_1-3,position_11-20,position_21-50,position_4-5,position_51+,position_6-10
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,8.783243,2.079442,0.001073,6.893301,0.693147,1,0,0,0,0,0,1
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,6.118097,0.000000,0.000000,3.214128,0.000000,0,0,0,0,1,0,0
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,8.636042,1.945910,0.001066,6.535346,1.386294,1,0,0,0,0,0,1
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,8.506132,2.639057,0.002629,7.435680,1.098612,1,0,0,0,0,0,1
4,client_73cda7b4e4f265ea,content_1855a661b4d36130,6.063785,0.693147,0.002331,3.871795,1.098612,1,0,0,0,1,0,0
5,client_73cda7b4e4f265ea,content_5d412fba6e1a2582,5.411646,0.693147,0.004484,10.538117,1.098612,1,0,1,0,0,0,0
6,client_73cda7b4e4f265ea,content_22c063002b7c1caf,5.752573,0.693147,0.003185,10.057325,0.000000,0,0,1,0,0,0,0
7,client_73cda7b4e4f265ea,content_aafb2ab7e5fc80d0,8.950273,3.044522,0.002594,5.127643,2.564949,1,0,0,0,0,0,1
8,client_73cda7b4e4f265ea,content_20403327d8d9374c,8.178077,2.397895,0.002808,9.273799,3.135494,1,0,0,0,0,0,1
9,client_73cda7b4e4f265ea,content_f71459b346aba398,5.170484,0.693147,0.005714,38.560000,0.000000,0,0,0,1,0,0,0


In [8]:
assert feature_vector.isna().sum().sum() == 0
assert feature_vector[["client_id", "content_id"]].duplicated().sum() == 0
assert len(feature_vector) == len(raw_features)

print("PASS: Feature vector has no missing values or duplicate units.")
print("Model features:")
for column in X.columns:
    print("-", column)

PASS: Feature vector has no missing values or duplicate units.
Model features:
- log_impressions_31d
- log_clicks_31d
- ctr_31d
- weighted_position_31d
- log_sessions_31d
- has_ga4_data
- position_1-3
- position_11-20
- position_21-50
- position_4-5
- position_51+
- position_6-10


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*


For each feature, the table documents its meaning, missing-value handling, categorical handling, and whether it exists before prediction.

In [9]:
feature_notes = pd.DataFrame([
    {
        "feature": "log_impressions_31d",
        "meaning": "Log transform of March GSC impressions.",
        "missing_handling": "Source nulls become 0; log1p(0)=0.",
        "categorical_handling": "Numeric.",
        "available_before_prediction": True,
    },
    {
        "feature": "log_clicks_31d",
        "meaning": "Log transform of March GSC clicks.",
        "missing_handling": "Source nulls become 0; log1p(0)=0.",
        "categorical_handling": "Numeric.",
        "available_before_prediction": True,
    },
    {
        "feature": "ctr_31d",
        "meaning": "March clicks divided by March impressions.",
        "missing_handling": "Filled with 0 when CTR cannot be calculated.",
        "categorical_handling": "Numeric.",
        "available_before_prediction": True,
    },
    {
        "feature": "weighted_position_31d",
        "meaning": "March impression-weighted average search position.",
        "missing_handling": "Filled with the observed March median.",
        "categorical_handling": "Numeric.",
        "available_before_prediction": True,
    },
    {
        "feature": "log_sessions_31d",
        "meaning": "Log transform of March GA4 sessions.",
        "missing_handling": "Source nulls become 0; log1p(0)=0.",
        "categorical_handling": "Numeric.",
        "available_before_prediction": True,
    },
    {
        "feature": "has_ga4_data",
        "meaning": "Whether GA4 data was available during March.",
        "missing_handling": "Filled with 0.",
        "categorical_handling": "Binary numeric indicator.",
        "available_before_prediction": True,
    },
    {
        "feature": "position_*",
        "meaning": "One-hot encoded March weighted-position bucket.",
        "missing_handling": "Missing position uses the Unknown bucket.",
        "categorical_handling": "One-hot encoded.",
        "available_before_prediction": True,
    },
])

display(feature_notes)

assert feature_notes["available_before_prediction"].all()
print("PASS: Every documented feature exists before prediction.")

,feature,meaning,missing_handling,categorical_handling,available_before_prediction
0,log_impressions_31d,Log transform of March GSC impressions.,Source nulls become 0; log1p(0)=0.,Numeric.,True
1,log_clicks_31d,Log transform of March GSC clicks.,Source nulls become 0; log1p(0)=0.,Numeric.,True
2,ctr_31d,March clicks divided by March impressions.,Filled with 0 when CTR cannot be calculated.,Numeric.,True
3,weighted_position_31d,March impression-weighted average search position.,Filled with the observed March median.,Numeric.,True
4,log_sessions_31d,Log transform of March GA4 sessions.,Source nulls become 0; log1p(0)=0.,Numeric.,True
5,has_ga4_data,Whether GA4 data was available during March.,Filled with 0.,Binary numeric indicator.,True
6,position_*,One-hot encoded March weighted-position bucket.,Missing position uses the Unknown bucket.,One-hot encoded.,True


PASS: Every documented feature exists before prediction.


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

The checks below attack the feature vector for:

- Future-window fields
- Labels or targets
- Outcome proxies
- Existing product flags
- Identifiers inside the model matrix
- Private names, URLs, queries, keywords, or text

In [10]:
forbidden_terms = [
    "future",
    "next_",
    "after_",
    "label",
    "target",
    "outcome",
    "proxy",
    "refresh_flag",
    "quick_win",
    "product_flag",
    "recommendation",
    "client_name",
    "company_name",
    "domain",
    "url",
    "query",
    "keyword",
    "page_title",
    "content_text",
]

name_based_hits = [
    column
    for column in X.columns
    if any(term in column.lower() for term in forbidden_terms)
]

allowed_features = set(numeric_features) | set(categorical_features.columns)

unexpected_features = [
    column for column in X.columns
    if column not in allowed_features
]

identifier_hits = [
    column
    for column in ["client_id", "content_id"]
    if column in X.columns
]

print("Name-based leakage hits:", name_based_hits)
print("Unexpected features:", unexpected_features)
print("Identifiers inside X:", identifier_hits)

assert name_based_hits == []
assert unexpected_features == []
assert identifier_hits == []

print("PASS: No future, label-derived, product-flag, private, or identifier fields entered X.")

Name-based leakage hits: []
Unexpected features: []
Identifiers inside X: []
PASS: No future, label-derived, product-flag, private, or identifier fields entered X.


In [11]:
source_window = con.execute('''
    SELECT
        MIN(report_date) AS minimum_date,
        MAX(report_date) AS maximum_date,
        COUNT(*) AS source_rows
    FROM warehouse
    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
      AND gsc_data_available IS TRUE
''').df()

display(source_window)

maximum_date = pd.to_datetime(
    source_window.loc[0, "maximum_date"]
).date()

assert str(maximum_date) <= "2026-03-31"
print("PASS: Feature source does not extend beyond March 2026.")

,minimum_date,maximum_date,source_rows
0,2026-03-01,2026-03-31,3611061


PASS: Feature source does not extend beyond March 2026.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

- `client_hash_id` — identifier only; not a transferable behavioral signal.
- `content_hash_id` — identifier only; not a transferable behavioral signal.
- `report_date` — used only to enforce the historical window.
- `month` — partition helper rather than a page-quality signal.
- Future-window metrics — unavailable at prediction time.
- Labels and outcome proxies — would reveal the answer.
- Existing FlyRank flags — would make the model copy the current rule.
- Client names, domains, URLs, titles, queries, keywords, and page text — excluded for privacy and memorization risk.

In [12]:
excluded_fields = pd.DataFrame(
    [
        ("client_hash_id", "Identifier only; excluded from X."),
        ("content_hash_id", "Identifier only; excluded from X."),
        ("report_date", "Used only to enforce the historical window."),
        ("month", "Partition helper rather than a behavioral feature."),
        ("future-window metrics", "Unavailable at prediction time."),
        ("labels and proxies", "Would reveal the outcome."),
        ("existing product flags", "Would make the model copy a rule."),
        ("names/domains/URLs/queries/text", "Privacy and memorization risk."),
    ],
    columns=["excluded_field_or_group", "reason"],
)

display(excluded_fields)

OUTPUT_DIR = Path("work/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

receipt = {
    "assignment": "ML-05 feature vector and leakage/privacy check",
    "feature_rows": int(len(X)),
    "feature_columns": int(X.shape[1]),
    "missing_values": int(X.isna().sum().sum()),
    "name_based_leakage_hits": name_based_hits,
    "unexpected_features": unexpected_features,
    "identifier_hits": identifier_hits,
    "future_inputs_used": False,
    "product_flags_used": False,
    "private_text_used": False,
}

receipt_path = OUTPUT_DIR / "w03_feature_leakage_metrics.json"

with open(receipt_path, "w", encoding="utf-8") as file:
    json.dump(receipt, file, indent=2)

print("Metrics receipt written to:", receipt_path)
display(pd.Series(receipt, name="value").to_frame())

,excluded_field_or_group,reason
0,client_hash_id,Identifier only; excluded from X.
1,content_hash_id,Identifier only; excluded from X.
2,report_date,Used only to enforce the historical window.
3,month,Partition helper rather than a behavioral feature.
4,future-window metrics,Unavailable at prediction time.
5,labels and proxies,Would reveal the outcome.
6,existing product flags,Would make the model copy a rule.
7,names/domains/URLs/queries/text,Privacy and memorization risk.


Metrics receipt written to: work/outputs/w03_feature_leakage_metrics.json


,value
assignment,ML-05 feature vector and leakage/privacy check
feature_rows,101441
feature_columns,12
missing_values,0
name_based_leakage_hits,[]
unexpected_features,[]
identifier_hits,[]
future_inputs_used,False
product_flags_used,False
private_text_used,False


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [13]:
assert X.isna().sum().sum() == 0
assert name_based_hits == []
assert unexpected_features == []
assert identifier_hits == []
assert receipt_path.exists()

print("ML-05 COMPLETE")
print("Feature rows:", len(X))
print("Feature columns:", X.shape[1])
print("Receipt:", receipt_path)

ML-05 COMPLETE
Feature rows: 101441
Feature columns: 12
Receipt: work/outputs/w03_feature_leakage_metrics.json
